In [1]:
#########################
# Include Headers
#########################

using LinearAlgebra
using HDF5
using Arpack
using CairoMakie


include("../header/.src/circuit/brickwall.jl")
include("../header/.src/circuit/heisenberg.jl")
#include("../header/.src/circuit/time-crystals.jl")
include("../header/.src/hdf5/hdf5Mods.jl")
#include("../header/.src/plotmods/colorschemes_mods.jl")
#include("../header/.src/random_matrices/random_matrices.jl")
#include("../header/.src/measurements/projectors.jl")
include("../header/.src/info_lattice/info_lattice.jl")
include("../header/.src/lazadires_diagram/lazadires_diagram.jl")






#### attrs-author-and-generating-file for given L

# parameters

L=8
N=2^L
N_band=N

Itrnumb=200
thetalist=collect(0.0:0.05:0.65)


file=h5open("../data/hiesenberg_transition_ordpar_L$(L).hdf5","cw")

set_hdf5_attributes(file)

close(file)





#### attrs-METADATA

file=h5open("../data/hiesenberg_transition_ordpar_L$(L).hdf5","cw")
 attrs = HDF5.attributes(file)

 attrs["4. METADATA"]=["This file contains raw eigen data and information lattice of individual eigenstates of the XXZ unitary, we aim to establish the critical point of the ETH-MBL transition with the aim to establish the unitary as a candidate driving scheme for the fisher information based probe for measurment induced phase transition as proposed by Arnau, Silvia and Xhek.
 
 We save the realistion of h, J for each realisation and since this is the only source of random-ness, results can be exactly reproduced. Relevant functions used to generate the circuit will be provided as a script. info-lattice Data for various projected eigenstates to be added in a seperate file. We also record benchmark times for the Arpack.eigen() and computation of info_lattice seperately. The phase-space scan is run after running a pre-compilation. benchmarktime for pre-compilation is also provided for reference."]



close(file)




#### attrs-model

file=h5open("../data/hiesenberg_transition_ordpar_L$(L).hdf5","cw")
 attrs = HDF5.attributes(file)

 attrs["[Model] 1. Unitary"]="exp(- i J_i ZZ + \theta/2 (XX+YY))exp(- i h_i Z)"

 attrs["[Model] 2. L"]=L

 attrs["[Model] 3. Tuning parameter"]="theta"
 
 attrs["[Model] 4. Range of J"]="[0, pi], Uniform Sampling"

 attrs["[Model] 5. Range of h"]="[0, 2pi], Uniform Sampling"

 attrs["[Model] 6. Order parameters"]="eigenvalues, info_lattice"
 
 attrs["[Model] 7. Range of theta"]="0.0:0.05:0.65"

 attrs["[Model] 8. Itrnumb"] = Itrnumb

 attrs["[Model] 9. Band Size for Info Lattice"] = N_band

close(file)



##########################################
# benchmark compile() step
##########################################


thetatry=0.4                    # rougly at the critical state
A=circuit_heisenberg(L, thetatry)


file=h5open("../data/hiesenberg_transition_ordpar_L$(L).hdf5","cw")

# benchmarking eigen

Bnch_T1_eig=Dates.now()
    eigvals,eigvecs= eigen(A)
Bnch_T2_eig=Dates.now()

file["L$(L)/compile/benchmark_time/eigen"]=string(Bnch_T2_eig-Bnch_T1_eig)



# sorting the eigenvalues to get a patch:

eigvals,eigvecs =phase_ordered_eigvecs(eigvals,eigvecs)

# benchmarking info_lattice

Bnch_T1_ilat=Dates.now()
        for i in 1:N_band
                state=eigvecs[i,:]
                info_lattice_state=info_lattice(state)
        end
Bnch_T2_ilat=Dates.now()

file["L$(L)/compile/benchmark_time/info_lattice"]=string(Bnch_T2_ilat-Bnch_T1_ilat)

close(file)




##################################################################
# Script for collecting eigen-data
##################################################################

  
  for itr in 1:Itrnumb
  
        for theta in thetalist

        
        file=h5open("../data/hiesenberg_transition_ordpar_L$(L).hdf5","cw")

        ## Drawing the Disorders

        
        J=rand(L-1)*pi;
        h=rand(L)*2*pi;


        ## Writing the disorder strengths

        file["L$(L)/theta$(theta)/Itr$(itr)/h"]=h;
        file["L$(L)/theta$(theta)/Itr$(itr)/J"]=J;

        
        ## Building Ckt

        A=circuit_heisenberg(L,theta,h,J)


        ## ED

        Bnch_T1_eig=Dates.now()
                eigvals,eigvecs= eigen(A)
        Bnch_T2_eig=Dates.now()

        
        # sorting the eigenvalues to get a patch:

        eigvals,eigvecs =phase_ordered_eigvecs(eigvals,eigvecs)


        ## Writing eigenvalues, benchmark_time

        file["L$(L)/theta$(theta)/Itr$(itr)/eigvals"]=eigvals;
        file["L$(L)/theta$(theta)/Itr$(itr)/benchmark_time/eigen"]=string(Bnch_T2_eig-Bnch_T1_eig)



        ## Writing info-lattice of individual eigenstates, and total benchmark_time

       
       
        Bnch_T1_ilat=Dates.now()

        for i in 1:N_band
                state=eigvecs[i,:]
                info_lattice_state=info_lattice(state)
                file["L$(L)/theta$(theta)/Itr$(itr)/info_lattice/eigvec_$(i)"]=lattice_to_vec(info_lattice(state))
        end

        Bnch_T2_ilat=Dates.now()

        
        
        file["L$(L)/theta$(theta)/Itr$(itr)/benchmark_time/info_lattice"]=string(Bnch_T2_ilat-Bnch_T1_ilat)


        close(file)

        end
        print("$(itr) \n")
        GC.gc()
end

┌ Warning: Pkg.installed() is deprecated
└ @ Pkg C:\Users\Ritam\.julia\juliaup\julia-1.11.7+0.x64.w64.mingw32\share\julia\stdlib\v1.11\Pkg\src\Pkg.jl:785


1 
2 
3 
4 
5 
6 
7 
8 
9 
10 
11 
12 
13 
14 
15 
16 
17 
18 
19 
20 
21 
22 
23 
24 
25 
26 
27 
28 
29 
30 
31 
32 
33 
34 
35 
36 
37 
38 
39 
40 
41 
42 
43 
44 
45 
46 
47 
48 
49 
50 
51 
52 
53 
54 
55 
56 
57 
58 
59 
60 
61 
62 
63 
64 
65 
66 
67 
68 
69 
70 
71 
72 
73 
74 
75 
76 
77 
78 
79 
80 
81 
82 
83 
84 
85 
86 
87 
88 
89 
90 
91 
92 
93 
94 
95 
96 
97 
98 
99 
100 
101 
102 
103 
104 
105 
106 
107 
108 
109 
110 
111 
112 
113 
114 
115 
116 
117 
118 
119 
120 
121 
122 
123 
124 
125 
126 
127 
128 
129 
130 
131 
132 
133 
134 
135 
136 
137 
138 
139 
140 
141 
142 
143 
144 
145 
146 
147 
148 
149 
150 
151 
152 
153 
154 
155 
156 
157 
158 
159 
160 
161 
162 
163 
164 
165 
166 
167 
168 
169 
170 
171 
172 
173 
174 
175 
176 
177 
178 
179 
180 
181 
182 
183 
184 
185 
186 
187 
188 
189 
190 
191 
192 
193 
194 
195 
196 
197 
198 
199 
200 
